In [ ]:
# ============================================
# Dogs vs Cats Classification
# Transfer Learning with MobileNetV2
# Author: Keo (Farhan Bin Hossain)
# ============================================

# NOTE: Before running this notebook:
# 1. Download dataset from Kaggle:
#    kaggle datasets download -d salader/dogsvscats
# 2. Place data in ./data/train and ./data/test
# ============================================

# ==================== IMPORTS ====================
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from sklearn.metrics import confusion_matrix, classification_report

# ==================== CONFIGURATION ====================
TRAIN_DIR  = "./data/train"   # Update path if needed
TEST_DIR   = "./data/test"
IMG_SIZE   = (128, 128)
BATCH_SIZE = 32

# ==================== LOAD DATA ====================
train_datagen = ImageDataGenerator(rescale=1.0/255)
test_datagen  = ImageDataGenerator(rescale=1.0/255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print(f"Train: {train_data.samples} images")
print(f"Test:  {test_data.samples} images")
print(f"Classes: {train_data.class_indices}")

# ==================== VISUALIZE SAMPLES ====================
from PIL import Image
from pathlib import Path

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Dogs vs Cats - Sample Images', fontsize=16)

dog_imgs = list(Path(f'{TRAIN_DIR}/dogs').glob('*.jpg'))[:4]
cat_imgs = list(Path(f'{TRAIN_DIR}/cats').glob('*.jpg'))[:4]

for idx, img_path in enumerate(dog_imgs):
    img = Image.open(img_path)
    axes[0, idx].imshow(img)
    axes[0, idx].set_title('Dog')
    axes[0, idx].axis('off')

for idx, img_path in enumerate(cat_imgs):
    img = Image.open(img_path)
    axes[1, idx].imshow(img)
    axes[1, idx].set_title('Cat')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

# ==================== BUILD MODEL ====================
print("\n🔧 Building MobileNetV2 model...\n")

base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)
print("Model built!")

# ==================== PHASE 1: FEATURE EXTRACTION ====================
print("\n Phase 1: Feature Extraction (Base Frozen)\n")

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history1 = model.fit(
    train_data,
    epochs=5,
    validation_data=test_data,
    verbose=1
)

# ==================== PHASE 2: FINE-TUNING ====================
print("\n Phase 2: Fine-tuning (Last 20 Layers)\n")

base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(
    train_data,
    epochs=5,
    validation_data=test_data,
    verbose=1
)

# ==================== EVALUATE ====================
print("\n Evaluating model...\n")

test_loss, test_acc = model.evaluate(test_data, verbose=0)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss:     {test_loss:.4f}")

test_data.reset()
predictions = model.predict(test_data, verbose=0)
pred_classes = (predictions > 0.5).astype(int).reshape(-1)
true_classes = test_data.classes

# Confusion Matrix
cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cat', 'Dog'],
            yticklabels=['Cat', 'Dog'])
plt.title(f'Confusion Matrix - {test_acc*100:.2f}% Accuracy')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(true_classes, pred_classes,
                           target_names=['Cat', 'Dog']))

# ==================== TRAINING HISTORY ====================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=16, fontweight='bold')

all_acc     = history1.history['accuracy']     + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss    = history1.history['loss']         + history2.history['loss']
all_val_loss= history1.history['val_loss']     + history2.history['val_loss']

axes[0].plot(all_acc,     label='Train',      marker='o')
axes[0].plot(all_val_acc, label='Validation', marker='s')
axes[0].axvline(x=5, color='red', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(all_loss,     label='Train',      marker='o')
axes[1].plot(all_val_loss, label='Validation', marker='s')
axes[1].axvline(x=5, color='red', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ==================== SAVE MODEL ====================
os.makedirs('./model', exist_ok=True)
model.save('./model/dogs_vs_cats_mobilenetv2.keras')
print("\n Model saved!")